# 명시적 N/V 경제표현 + 개인별 경제적합도 M5: Dunnhumby test-only 10 seeds

Seed 42에서 실행한 M5-v2의 수식·6개 arm·rho=0.15·lambda=0.5·K=5 균등 음성·100 epoch을 변경하지 않고 seed 42~51로 확장합니다. 완료된 seed 42 test JSON은 재학습하거나 재평가하지 않고 검증 후 재사용하며, 실제 GPU 학습은 seed 43~51만 수행합니다. DAY 1~697을 학습하고 DAY 698~704 test를 seed·arm당 한 번만 평가합니다. validation·early stopping·holdout은 없습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '857302557f91c20aa8b4e9caa0b112c04f9dabfe'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_nv_economic_positive_weight_test import (
    FULL_SEEDS,
    configure_m5_nv_economic_positive_test_run,
    preflight_summary,
    run_m5_nv_economic_positive_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
PILOT_RESULT = '/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_explicit_nv_personalized_economic_positive_weighting_test_seed42_v2/m5_economic_positive_weight_test_dbef3f3aa752.json'
cfg = configure_m5_nv_economic_positive_test_run(
    dataset='dunnhumby',
    seeds=FULL_SEEDS,
    reused_seed42_json=PILOT_RESULT,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_explicit_nv_personalized_economic_positive_weighting_test_multiseed_v2',
)
summary = preflight_summary(cfg)
assert cfg.seeds == tuple(range(42, 52))
assert summary['seed42_handling'] == 'reuse the completed seed-42 test result; train seeds 43--51 only'
assert summary['validation_constructed'] is False
assert summary['holdout_evaluation'] is False
assert summary['m2']['q_n'] == 'post-projection strength gate only'
assert summary['m2']['q_c_used_in_m2'] is False
assert 'clipped_user_bin_fit' in summary['m4_prime']['formula']
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_m5_nv_economic_positive_test(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) seed별 test 절대지표')
show(result_df)
print('2) 10-seed 평균·표준편차·95% t 구간')
show(result_df.attrs['mean'])
print('3) seed별 대조군 비교')
show(result_df.attrs['comparison'])
print('4) 동일 seed 대응차 평균과 양수 seed 수')
show(result_df.attrs['paired_mean'])
print("5) seed별 M2 × M4 상호작용 — 보조 지표")
show(result_df.attrs['interaction'])
print('6) 사전 고정 기준의 seed별 기술적 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))